# Function calling viết tay: vòng gọi tool, kiểm tham số, gọi lại khi sai

Model không chạy được code của bạn: nó chỉ **đề nghị** gọi một hàm với một bộ tham số. Phần còn lại (kiểm tham số, gọi
hàm, gửi kết quả lại, dừng đúng lúc, không để model lặp vô tận) là code của bạn. Bài này viết phần đó không dùng
framework, cho một trợ lý trả lời câu hỏi thống kê về phản hồi sinh viên UIT-VSFC bằng ba tool, rồi so hai cách gọi
tool với `qwen2.5:3b`: **native** (trường `tools` của API, template của model lo định dạng) và **prompted** (mô tả tool
trong system prompt, ép output JSON bằng schema; chạy được với mọi LLM qua `llmkit`).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp, hằng. Điền vào chỗ `# TODO`. Giữ nguyên `TOOLS`, `PROMPTED_SCHEMA`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu, gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/11-llm-apis-prompting/exercises/intermediate/01-function-calling -q`

In [ ]:
# explore: tìm gốc repo, tải UIT-VSFC nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

VSFC = ensure("uit-vsfc")
print(VSFC)

In [ ]:
import json
import time
from dataclasses import dataclass, field
from typing import Callable

import jsonschema
import pandas as pd

from llmkit import LLMClient

## 1. Ba tool và registry

`TOOLS` mô tả tool theo dạng **OpenAI Chat Completions** (mục 6 lý thuyết): mỗi tool là một `function` có `name`,
`description` và `parameters` là JSON Schema. Mô tả là thứ model đọc để quyết định gọi gì: viết nó như tài liệu cho
một đồng nghiệp, kể cả ý nghĩa từng giá trị enum.

Dữ liệu là một `DataFrame` có cột `sentence`, `sentiment` (0, 1, 2 lần lượt là negative, neutral, positive) và `topic`
(0 lecturer, 1 training_program, 2 facility, 3 others), như tập test của UIT-VSFC.

`make_registry(df) -> dict[str, Callable]`: ba hàm Python nhận đúng tham số theo tên trong `TOOLS` và trả **dict**:

| tool | trả về |
| --- | --- |
| `count_feedback(topic=None, sentiment=None)` | `{"count": số dòng khớp}`; tham số vắng mặt nghĩa là không lọc theo nó |
| `percentage(part, whole)` | `{"percent": round(100 * part / whole, 1)}`; `whole` bằng 0 thì ném `ValueError` |
| `list_examples(topic, sentiment, limit=3)` | `{"sentences": [...]}`: `limit` câu **đầu tiên** theo thứ tự trong `df` |

In [ ]:
TOPICS = ["lecturer", "training_program", "facility", "others"]
SENTIMENTS = ["negative", "neutral", "positive"]
TOPIC_DOC = "Chủ đề: lecturer (giảng viên), training_program (chương trình đào tạo), facility (cơ sở vật chất), others (khác)."
SENTIMENT_DOC = "Cảm xúc: negative (tiêu cực), neutral (trung tính), positive (tích cực)."

TOOLS = [
    {"type": "function", "function": {
        "name": "count_feedback",
        "description": "Đếm số phản hồi sinh viên trong tập test UIT-VSFC, lọc theo chủ đề và/hoặc cảm xúc. "
                       "Bỏ một tham số nghĩa là không lọc theo nó. " + TOPIC_DOC + " " + SENTIMENT_DOC,
        "parameters": {"type": "object", "additionalProperties": False, "properties": {
            "topic": {"type": "string", "enum": TOPICS},
            "sentiment": {"type": "string", "enum": SENTIMENTS}}},
    }},
    {"type": "function", "function": {
        "name": "percentage",
        "description": "Tính part / whole * 100, làm tròn 1 chữ số thập phân. Dùng sau khi đã có hai số đếm.",
        "parameters": {"type": "object", "additionalProperties": False, "required": ["part", "whole"], "properties": {
            "part": {"type": "number", "minimum": 0},
            "whole": {"type": "number", "minimum": 0}}},
    }},
    {"type": "function", "function": {
        "name": "list_examples",
        "description": "Lấy vài câu phản hồi mẫu theo chủ đề và cảm xúc. " + TOPIC_DOC + " " + SENTIMENT_DOC,
        "parameters": {"type": "object", "additionalProperties": False, "required": ["topic", "sentiment"],
                       "properties": {
                           "topic": {"type": "string", "enum": TOPICS},
                           "sentiment": {"type": "string", "enum": SENTIMENTS},
                           "limit": {"type": "integer", "minimum": 1, "maximum": 5}}},
    }},
]

In [ ]:
def make_registry(df: pd.DataFrame) -> dict[str, Callable]:
    """{"count_feedback": ..., "percentage": ..., "list_examples": ...} chạy trên df (bảng ở trên)."""

    def select(topic=None, sentiment=None) -> pd.DataFrame:
        mask = pd.Series(True, index=df.index)
        if topic is not None:
            mask &= df.topic == TOPICS.index(topic)
        if sentiment is not None:
            mask &= df.sentiment == SENTIMENTS.index(sentiment)
        return df[mask]

    def count_feedback(topic=None, sentiment=None) -> dict:
        return {"count": int(len(select(topic, sentiment)))}

    def percentage(part, whole) -> dict:
        if whole == 0:
            raise ValueError("whole bằng 0")
        return {"percent": round(100 * part / whole, 1)}

    def list_examples(topic, sentiment, limit=3) -> dict:
        return {"sentences": select(topic, sentiment).sentence.head(limit).tolist()}

    return {"count_feedback": count_feedback, "percentage": percentage, "list_examples": list_examples}

## 2. Thực thi một tool call và vòng lặp

Một tool call theo dạng OpenAI: `{"id": "call_1", "type": "function", "function": {"name": ..., "arguments": ...}}`;
`arguments` là **chuỗi** JSON (OpenAI) hoặc **dict** (Ollama), nhận cả hai.

`execute_tool_call(call, tools, registry) -> str`: nội dung gửi lại cho model, **không bao giờ ném lỗi**:

- thành công: `json.dumps(kết quả, ensure_ascii=False)`;
- lỗi thì một chuỗi bắt đầu bằng `"ERROR: "` rồi mô tả đủ để model tự sửa: tên tool không có (kèm danh sách tool
  hợp lệ), arguments không phải JSON object, arguments sai schema (kiểm bằng `jsonschema` với `parameters` của tool,
  nêu thông điệp lỗi), hoặc hàm ném exception (nêu thông điệp).

`run_tool_loop(chat, messages, tools, registry, max_steps=5) -> LoopResult`:

1. Gọi `chat(history, tools)`; nó trả một message assistant dạng OpenAI (`content`, `tool_calls` có thể vắng hoặc rỗng).
   Mỗi lần gọi là một **bước**.
2. Không có tool call: dừng, `answer` là `content`, `stopped="answer"`.
3. Có tool call: tool call nào thiếu `id` thì đặt `f"call_{bước}_{thứ tự}"` (bước và thứ tự đếm từ 0); thêm message
   assistant (đã có id) vào lịch sử, thực thi **từng** call theo thứ tự, thêm mỗi kết quả một message
   `{"role": "tool", "tool_call_id": id, "name": tên, "content": kết quả}`; ghi vào `tool_calls` của kết quả
   `(tên, arguments đã parse hoặc nguyên văn nếu hỏng, nội dung kết quả)`; kết quả bắt đầu bằng `ERROR: ` thì `errors`
   tăng 1. Quay lại bước 1.
4. Đã gọi `chat` đủ `max_steps` lần mà lần cuối vẫn đòi tool: dừng với `answer=None`, `stopped="max_steps"` (tool của
   lần cuối vẫn được thực thi).

`messages` của `LoopResult` là toàn bộ lịch sử (bắt đầu bằng bản sao của `messages` đầu vào, không sửa đầu vào).

In [ ]:
@dataclass
class LoopResult:
    answer: str | None
    messages: list[dict]
    steps: int
    tool_calls: list[tuple[str, object, str]] = field(default_factory=list)
    errors: int = 0
    stopped: str = "answer"            # "answer" | "max_steps"


def _parse_arguments(raw):
    if isinstance(raw, dict):
        return raw
    try:
        value = json.loads(raw)
    except (TypeError, ValueError):
        return None
    return value if isinstance(value, dict) else None


def execute_tool_call(call: dict, tools: list[dict], registry: dict[str, Callable]) -> str:
    """Kết quả (JSON) hoặc "ERROR: ..."; không bao giờ ném lỗi."""
    specs = {t["function"]["name"]: t["function"] for t in tools}
    name = (call.get("function") or {}).get("name")
    if name not in specs or name not in registry:
        return f"ERROR: không có tool {name!r}. Tool hợp lệ: {', '.join(sorted(specs))}."
    raw = call["function"].get("arguments", {})
    args = _parse_arguments(raw)
    if args is None:
        return f"ERROR: arguments của {name} phải là một JSON object, nhận {raw!r}."
    problems = [e.message for e in jsonschema.Draft202012Validator(specs[name]["parameters"]).iter_errors(args)]
    if problems:
        return f"ERROR: arguments của {name} sai schema: " + "; ".join(problems)
    try:
        result = registry[name](**args)
    except Exception as exc:
        return f"ERROR: {name} lỗi khi chạy: {type(exc).__name__}: {exc}"
    return json.dumps(result, ensure_ascii=False)


def run_tool_loop(chat: Callable[[list[dict], list[dict]], dict], messages: list[dict], tools: list[dict],
                  registry: dict[str, Callable], max_steps: int = 5) -> LoopResult:
    """Vòng gọi tool tới khi model trả lời hoặc hết max_steps."""
    result = LoopResult(answer=None, messages=[dict(m) for m in messages], steps=0)
    for step in range(max_steps):
        reply = chat(result.messages, tools)
        result.steps += 1
        calls = reply.get("tool_calls") or []
        if not calls:
            result.answer = reply.get("content") or ""
            result.messages.append({"role": "assistant", "content": result.answer})
            result.stopped = "answer"
            return result
        calls = [{**c, "id": c.get("id") or f"call_{step}_{i}"} for i, c in enumerate(calls)]
        result.messages.append({"role": "assistant", "content": reply.get("content"), "tool_calls": calls})
        for call in calls:
            content = execute_tool_call(call, tools, registry)
            name = (call.get("function") or {}).get("name")
            raw = (call.get("function") or {}).get("arguments")
            parsed = _parse_arguments(raw)
            result.tool_calls.append((name, parsed if parsed is not None else raw, content))
            result.errors += content.startswith("ERROR: ")
            result.messages.append({"role": "tool", "tool_call_id": call["id"], "name": name, "content": content})
    result.stopped = "max_steps"
    return result

## 3. Tool calling bằng prompt cho LLM không có `tools`

`llmkit.chat` (và nhiều model, server) chỉ nhận và trả **chuỗi**. `prompted_chat(llm)` trả một hàm
`chat(messages, tools) -> message assistant dạng OpenAI` dùng được với `run_tool_loop`, bằng cách:

1. Thêm **một message system đứng đầu** mô tả giao thức: liệt kê mỗi tool một dòng gồm tên, mô tả và
   `json.dumps(parameters, ensure_ascii=False)`; nói model trả **JSON** `{"tool_calls": [{"name": ..., "arguments":
   {...}}]}` khi cần gọi tool, hoặc `{"answer": "..."}` khi đã đủ thông tin trả lời. Giữ nguyên các message system
   của người gọi ngay sau nó.
2. Đổi lịch sử sang dạng chỉ có chuỗi: message assistant có `tool_calls` thành `content` là
   `json.dumps({"tool_calls": [{"name": ..., "arguments": ...}]}, ensure_ascii=False)`; message `tool` thành message
   **user** `"Kết quả tool <name> (<tool_call_id>): <content>"`. Message khác giữ nguyên `role`, `content`.
3. Gọi `llm.chat(history, json_schema=PROMPTED_SCHEMA)` một lần.
4. Đọc JSON trả về: `tool_calls` khác rỗng thì trả `{"role": "assistant", "content": None, "tool_calls": [{"type":
   "function", "function": {"name": ..., "arguments": {...}}}, ...]}` (không có `id`: `run_tool_loop` tự đặt); ngược lại
   trả `{"role": "assistant", "content": answer}`. Câu trả lời không phải JSON object thì coi **cả chuỗi** là câu trả
   lời cuối.

In [ ]:
PROMPTED_SCHEMA = {
    "type": "object",
    "properties": {
        "tool_calls": {"type": "array", "items": {
            "type": "object", "required": ["name", "arguments"],
            "properties": {"name": {"type": "string"}, "arguments": {"type": "object"}}}},
        "answer": {"type": "string"},
    },
}


def _protocol(tools: list[dict]) -> str:
    lines = ["Bạn có các tool sau (tên: mô tả. Tham số theo JSON Schema):"]
    for tool in tools:
        f = tool["function"]
        lines.append(f"- {f['name']}: {f['description']} Tham số: {json.dumps(f['parameters'], ensure_ascii=False)}")
    lines += [
        "Mỗi lượt, chỉ trả lời bằng MỘT JSON object:",
        '- cần dữ liệu: {"tool_calls": [{"name": "<tên tool>", "arguments": {...}}]} (có thể gọi nhiều tool một lượt);',
        '- đã đủ thông tin: {"answer": "<câu trả lời tiếng Việt, nêu rõ con số>"}.',
        "Không tự bịa con số: số liệu phải lấy từ kết quả tool. Câu hỏi không liên quan tới tool thì trả lời thẳng.",
    ]
    return "\n".join(lines)


def prompted_chat(llm: LLMClient) -> Callable[[list[dict], list[dict]], dict]:
    """Bọc một LLM chỉ-có-chuỗi thành hàm chat(messages, tools) -> message assistant dạng OpenAI."""

    def chat(messages: list[dict], tools: list[dict]) -> dict:
        history = [{"role": "system", "content": _protocol(tools)}]
        for m in messages:
            if m["role"] == "assistant" and m.get("tool_calls"):
                calls = [{"name": c["function"]["name"], "arguments": c["function"]["arguments"]}
                         for c in m["tool_calls"]]
                history.append({"role": "assistant",
                                "content": json.dumps({"tool_calls": calls}, ensure_ascii=False)})
            elif m["role"] == "tool":
                history.append({"role": "user",
                                "content": f"Kết quả tool {m.get('name')} ({m['tool_call_id']}): {m['content']}"})
            else:
                history.append({"role": m["role"], "content": m["content"]})
        reply = llm.chat(history, json_schema=PROMPTED_SCHEMA)
        try:
            data = json.loads(reply)
        except ValueError:
            data = None
        if not isinstance(data, dict):
            return {"role": "assistant", "content": reply}
        calls = data.get("tool_calls") or []
        if calls:
            return {"role": "assistant", "content": None, "tool_calls": [
                {"type": "function", "function": {"name": c.get("name"), "arguments": c.get("arguments", {})}}
                for c in calls]}
        return {"role": "assistant", "content": data.get("answer", "")}

    return chat

## 4. Chạy thật: native và prompted trên 12 câu hỏi

Mười câu hỏi có đáp án số tính được từ tập test (một hoặc hai lần đếm, có câu cần thêm `percentage`), một câu cần
`list_examples`, một câu ngoài phạm vi ("Hôm nay Hà Nội có mưa không?") mà model **không được** gọi tool. Chấm:

- câu có số: mọi con số đáp án xuất hiện trong câu trả lời (phần trăm sai lệch tối đa 0.1);
- câu `list_examples`: có ít nhất một lần gọi `list_examples` với `topic="facility"`, `sentiment="negative"`;
- câu ngoài phạm vi: không gọi tool nào.

**Native**: gửi `tools` thẳng cho `/api/chat` của Ollama; template của Qwen2.5 tự chèn mô tả tool và đọc thẻ
`<tool_call>` model sinh ra thành `message.tool_calls`. **Prompted**: `prompted_chat(llmkit.get_client())`, tức
`llmkit.chat` với `json_schema`. Cả hai đi qua cùng `run_tool_loop`. Khoảng 50 lượt gọi model, dưới 2 phút trên PC
của track.

In [ ]:
# explore: dữ liệu, registry, bộ câu hỏi có đáp án tính từ dữ liệu
test = pd.read_parquet(VSFC / "default" / "test" / "0000.parquet")
registry = make_registry(test)
count = lambda **kw: registry["count_feedback"](**kw)["count"]
pct = lambda a, b: round(100 * a / b, 1)

QUESTIONS = [
    ("Tập test có tất cả bao nhiêu phản hồi?", [count()]),
    ("Có bao nhiêu phản hồi tiêu cực về cơ sở vật chất?", [count(topic="facility", sentiment="negative")]),
    ("Có bao nhiêu phản hồi tích cực?", [count(sentiment="positive")]),
    ("Có bao nhiêu phản hồi về chương trình đào tạo?", [count(topic="training_program")]),
    ("Chủ đề khác (others) có bao nhiêu phản hồi trung tính?", [count(topic="others", sentiment="neutral")]),
    ("Có bao nhiêu phản hồi tích cực và bao nhiêu phản hồi tiêu cực về giảng viên?",
     [count(topic="lecturer", sentiment="positive"), count(topic="lecturer", sentiment="negative")]),
    ("Phản hồi tiêu cực chiếm bao nhiêu phần trăm trong các phản hồi về cơ sở vật chất?",
     [pct(count(topic="facility", sentiment="negative"), count(topic="facility"))]),
    ("Phản hồi trung tính chiếm bao nhiêu phần trăm toàn bộ tập test?", [pct(count(sentiment="neutral"), count())]),
    ("Trong các phản hồi về chương trình đào tạo, tỉ lệ phản hồi tiêu cực là bao nhiêu phần trăm?",
     [pct(count(topic="training_program", sentiment="negative"), count(topic="training_program"))]),
    ("Bao nhiêu phần trăm phản hồi tích cực là nói về giảng viên?",
     [pct(count(topic="lecturer", sentiment="positive"), count(sentiment="positive"))]),
    ("Cho tôi xem 2 phản hồi tiêu cực về cơ sở vật chất.", "list_examples"),
    ("Hôm nay Hà Nội có mưa không?", "no_tool"),
]
for q, gold in QUESTIONS:
    print(gold, "|", q)

In [ ]:
# explore: hai cách gọi tool, cùng vòng lặp, cùng câu hỏi
import re

import httpx

import llmkit

SYSTEM = {"role": "system", "content": "Bạn là trợ lý phân tích phản hồi sinh viên UIT-VSFC. Trả lời tiếng Việt, "
                                       "ngắn gọn, nêu rõ con số lấy từ tool."}


def native_chat(messages: list[dict], tools: list[dict]) -> dict:
    """Ollama /api/chat với trường tools; đổi message về dạng Ollama (arguments là object) và ngược lại."""
    wire = []
    for m in messages:
        m = dict(m)
        if m.get("tool_calls"):
            m["tool_calls"] = [{"function": {"name": c["function"]["name"],
                                             "arguments": c["function"]["arguments"]}} for c in m["tool_calls"]]
        if m["role"] == "tool":
            m = {"role": "tool", "content": m["content"], "tool_name": m.get("name")}
        wire.append(m)
    r = httpx.post("http://localhost:11434/api/chat", timeout=300, json={
        "model": "qwen2.5:3b", "messages": wire, "tools": tools, "stream": False,
        "options": {"temperature": 0, "seed": 0}}).json()
    msg = r["message"]
    calls = [{"type": "function", "function": c["function"]} for c in msg.get("tool_calls") or []]
    return {"role": "assistant", "content": msg.get("content", ""), "tool_calls": calls}


def numbers_in(text: str) -> list[float]:
    """Số trong câu tiếng Việt: '1.590' và '1 590' là nghìn, '95,2' là thập phân."""
    out = []
    for raw in re.findall(r"\d+(?:[ .,]\d+)*", text or ""):
        if re.fullmatch(r"\d{1,3}(?:[ .]\d{3})+", raw):
            out.append(float(re.sub(r"[ .]", "", raw)))
        else:
            out.append(float(raw.replace(",", ".").replace(" ", "")) if raw.count(",") + raw.count(".") <= 1
                       else float(re.sub(r"[ .,]", "", raw)))
    return out


def correct(result: LoopResult, gold) -> bool:
    if gold == "no_tool":
        return not result.tool_calls
    if gold == "list_examples":
        return any(name == "list_examples" and isinstance(args, dict) and args.get("topic") == "facility"
                   and args.get("sentiment") == "negative" for name, args, _ in result.tool_calls)
    found = numbers_in(result.answer)
    return all(any(abs(f - g) <= 0.1 for f in found) for g in gold)


modes = {"native": native_chat, "prompted": prompted_chat(llmkit.get_client())}
rows = []
for mode, chat in modes.items():
    for q, gold in QUESTIONS:
        t0 = time.perf_counter()
        res = run_tool_loop(chat, [SYSTEM, {"role": "user", "content": q}], TOOLS, registry, max_steps=5)
        rows.append(dict(mode=mode, question=q, ok=correct(res, gold), steps=res.steps, calls=len(res.tool_calls),
                         errors=res.errors, stopped=res.stopped, seconds=time.perf_counter() - t0, answer=res.answer,
                         trace=[(n, a) for n, a, _ in res.tool_calls]))
        print(f"[{mode:8s}] {'ĐÚNG' if rows[-1]['ok'] else 'sai '} {res.steps} bước {rows[-1]['seconds']:4.1f}s | {q}")
        print(f"           tool: {rows[-1]['trace']}")
        print(f"           đáp: {(res.answer or '')[:160]!r}")

table = pd.DataFrame(rows)
print(table.groupby("mode").agg(dung=("ok", "sum"), buoc_tb=("steps", "mean"), goi_tool=("calls", "sum"),
                                loi_tool=("errors", "sum"), giay_tb=("seconds", "mean")).round(2))

In [ ]:
# explore: model nhận lỗi và tự sửa? Gửi một tool call sai schema rồi xem lượt sau
broken = [SYSTEM, {"role": "user", "content": "Có bao nhiêu phản hồi tiêu cực về cơ sở vật chất?"},
          {"role": "assistant", "content": None, "tool_calls": [{"id": "call_x", "type": "function", "function": {
              "name": "count_feedback", "arguments": {"topic": "phòng học", "sentiment": "negative"}}}]}]
broken.append({"role": "tool", "tool_call_id": "call_x", "name": "count_feedback",
               "content": execute_tool_call(broken[-1]["tool_calls"][0], TOOLS, registry)})
print(broken[-1]["content"])
for mode, chat in modes.items():
    res = run_tool_loop(chat, broken, TOOLS, registry, max_steps=3)
    print(f"{mode}: lượt sau gọi {[(n, a) for n, a, _ in res.tool_calls]} -> {(res.answer or '')[:120]!r}")

## Đọc kết quả

So với bảng ở mục 6 lý thuyết. Tự trả lời:

- Cách nào đúng nhiều câu hơn trên 12 câu? Với 12 câu, chênh một hai câu có nói được gì không (bài Intermediate 02)?
- Câu ngoài phạm vi: cách nào vẫn gọi tool? Câu phần trăm: model có gọi `percentage` hay tự tính (và tính đúng không)?
- Có lượt nào tool trả `ERROR:`? Ở thí nghiệm cuối, model có tự sửa tham số sau khi đọc thông báo lỗi không?
- Prompted tốn bao nhiêu token system cho mô tả tool so với native (gợi ý: in `prompt_eval_count` của một lượt)?

Sau đó làm phần trắc nghiệm của bài trong app.